In [ ]:
import pandas as pd
import glob, os
from tqdm.notebook import tqdm
import json
import plotly.express as px
import plotly.graph_objects as go
import geopandas as gpd
import numpy as np

pd.set_option('display.max_columns', None)

In [ ]:
CODIGO_COMUNIDAD = 28
ELECCIONES = 201111
CODIGO_MUNICIPIO=5
ELECCIONES_FILES_PATH = "../../data/raw/resultados_electorales/congreso_por_municipio/"
ELECCIONES_COMPLETAS_PATH = "../../data/staging/elecciones_generales_completas.csv"
GEOJSON_PATH = "../../data/raw/mapa_municipios/Municipios_IGN-shp/madrid_municipios_reduced.geojson"
PRECALCULATED_TABLE = True 

# Datos electorales

In [ ]:
def clean_column_names(df):
    df.columns = df.columns.str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace(':', '').str.replace('(', '')
    df.columns = df.columns.str.replace('.', '').str.replace(')', '').str.replace('-', '_').str.replace(')', '')
    df.columns = df.columns.str.replace("á", "a").str.replace("é", "e").str.replace("í", "i").str.replace("ó", "o").str.replace("ú", "u")
    return df

def map_party_names(df):
    mappings={       
              'cs': 'ciudadanos', 
              "c's": 'ciudadanos',  
              "psoe_progr":"psoe",}
    df = df.rename(columns=mappings)
    return df

def load_data(data_path):
    df = None
    for file in tqdm(glob.glob(data_path + "*.xlsx")):
        df_single = pd.read_excel(file, header=1, skiprows=4)
        year = os.path.splitext(file)[0].split("/")[-1][3:7]
        month = os.path.splitext(file)[0].split("/")[-1][7:9]
        df_single["year"] = int(year)
        df_single["month"] = int(month)
        df_single["yearmonth"] = year+month
        print("yearmonth: ", year+month)
        df_single = clean_column_names(df_single)

        df_single = map_party_names(df_single)
        
#        print(df_single.columns)
        if df is None:
            df = df_single
        else:
            df = pd.concat([df, df_single],axis=0, ignore_index=True, sort=False)

    cols = [c for c in df.columns if "nnamed" not in c]
    df = df[cols]
    df = df.fillna(0)
    print(df.columns)
    return df

def add_custom_vars(df):
    # crear codigo ine
    print("adding CODIGOINE")
    df["CODIGOINE"] = df["codigo_de_provincia"].astype(int).astype(str).str.zfill(2) + df["codigo_de_municipio"].astype(int).astype(str).str.zfill(3)
    return df

In [ ]:
if not PRECALCULATED_TABLE:
    df = load_data(ELECCIONES_FILES_PATH)
    df = add_custom_vars(df)
    df.to_csv(ELECCIONES_COMPLETAS_PATH, sep="|")
else:
    df = pd.read_csv(ELECCIONES_COMPLETAS_PATH, sep="|", dtype={"CODIGOINE":str}, index_col=False)
    df = df.drop(columns=["Unnamed: 0"])

In [ ]:
key_columns = ['nombre_de_comunidad', 'codigo_de_provincia', 'nombre_de_provincia',
               'codigo_de_municipio', 'nombre_de_municipio', 'poblacion',
               'numero_de_mesas', 'total_censo_electoral', 'total_votantes',
               'votos_validos', 'votos_a_candidaturas','votos_en_blanco','votos_nulos',
               'year', 'month', 'yearmonth', 'CODIGOINE']
party_columns = [x for x in df.columns if x not in key_columns]

In [ ]:
df.shape

In [ ]:
df[party_columns] = df[party_columns].apply(pd.to_numeric)

## Filtrar por Madrid

In [ ]:
df_comunidad = df[df["codigo_de_provincia"] == CODIGO_COMUNIDAD]  # 28 = Madrid

In [ ]:
df_comunidad.shape

# Mapa Comunidad por munipios

In [ ]:
#map_path = "../../data/raw/mapa_municipios/Municipios_IGN-shp/Municipios_IGN.shp"
#geodf = gpd.read_file(map_path)
#geodf["COMUNIDAD"] = geodf["CODIGOINE"].apply(lambda x: str(x)[:2])
#geodf = geodf[geodf["COMUNIDAD"] == str(CODIGO_COMUNIDAD).zfill(2)]
#geodf_reduced = geodf[["CODIGOINE","geometry"]]
#geojson_save_path = "../../data/raw/mapa_municipios/Municipios_IGN-shp/madrid_municipios.geojson"
#guardar
#geodf_reduced.to_file(geojson_save_path, driver = "GeoJSON")

In [ ]:
#cargar
with open(GEOJSON_PATH) as geofile:
    geojson = json.load(geofile)

# Mapa de ganadores

In [ ]:
# creando una columna con el partido con más votos
df_comunidad["ganador"] = df_comunidad[party_columns].idxmax(axis=1)
df_comunidad["ganador"] = df_comunidad.apply(lambda x: None if x[party_columns].sum() == 0 else x["ganador"], axis=1)
df_comunidad = df_comunidad[df_comunidad["ganador"].notnull()]

In [ ]:
df_comunidad["ganador"].unique()

# Resultados electorales

In [ ]:
color_map = {'ucd':"#FFA07A",
             'psoe':"#E52B50", 
             'pp':"#1E90FF", 
             'ap':"#FFFF00",              
             'pce':"#FF0000",              
             'uce':"#FF0000",              
             'ap_pdp':"#FFFF00",              
             'pcoe':"#FF0000",              
             'pst':"#FF0000",              
             'ap_pdp_pl':"#FFFF00",              
             'cds':"#66CDAA",              
             'iu':"#32CD32",
             'fe':"#000000", 
             'ciudadanos':"#FFA500",
             'podemos_iu':"#9370DB",    
             'podemos_iu_equo':"#9370DB",                 
             'podemos':"#9370DB",
             'vox':"#00FF00"}

In [ ]:
# filtros
#df_com_elecciones = df_comunidad[df_comunidad["yearmonth"] == ELECCIONES]
df_com_elecciones = df_comunidad.copy()
df_com_elecciones = df_com_elecciones[["ganador", "nombre_de_municipio", "CODIGOINE", "year", "month"]].sort_values(["year", "month"], ascending=True)

In [ ]:
df_com_elecciones.shape

In [ ]:
df_com_elecciones.tail()

# Gráficos

In [ ]:
#Forzar ganadores en el primer frame para que funcione
df_forze_winner = pd.DataFrame({"ganador":df_com_elecciones.ganador.unique(), 
              "nombre_de_municipio":["UNKNOWN"] * len(df_com_elecciones.ganador.unique()),
              "CODIGOINE":["UNKNOWN"] * len(df_com_elecciones.ganador.unique()),
              "year":[1977] * len(df_com_elecciones.ganador.unique()),
              "month":[6] * len(df_com_elecciones.ganador.unique())}) 
df_tmp = df_com_elecciones.append(df_forze_winner, ignore_index = True)

In [ ]:
## GRAFICO --------------------
fig = px.choropleth_mapbox(data_frame=df_tmp, geojson=geojson, locations='CODIGOINE', 
                           featureidkey="properties.CODIGOINE", color='ganador', mapbox_style="carto-positron", 
                           hover_data=["nombre_de_municipio", "ganador", "year", "month"],  
                           color_discrete_map=color_map, zoom=7,
                           center={"lat": 40.463667, "lon": -3.74922}, animation_frame="year",
                           animation_group="ganador",height=650,
                           title= f"Ganador por municipio de las Elecciones Generales en la Comunidad de Madrid")
fig.show()

In [ ]:
## GRAFICO --------------------
# una única elección
df_tmp = df_com_elecciones.copy()
df_tmp["yearmonth"] = df_tmp["year"].astype(str).str.zfill(4) + df_tmp["month"].astype(str).str.zfill(2)
df_tmp = df_tmp[df_tmp["yearmonth"] == str(ELECCIONES)]
fig = px.choropleth_mapbox(data_frame=df_tmp, 
                           geojson=geojson, locations='CODIGOINE', 
                           featureidkey="properties.CODIGOINE", color='ganador', mapbox_style="carto-positron", 
                           hover_data=["nombre_de_municipio", "ganador"], zoom=7, color_discrete_map=color_map,
                           center={"lat": 40.463667, "lon": -3.74922}, 
                           title= f"Elecciones Generales {str(ELECCIONES)[:4]}-{str(ELECCIONES)[-2:]} en Madrid")
fig.show()

# Evolución votos

In [ ]:
# evolucion partidos mas votados, lineas dejar rastro, filtro por municipio

In [ ]:
df_municipio = df_comunidad[df_comunidad.codigo_de_municipio == CODIGO_MUNICIPIO]
nombre_municipio = df_municipio.nombre_de_municipio.unique()[0]
df_municipio.head()

In [ ]:
df_party_series = df_municipio.melt(id_vars=['year', "month"], value_vars=party_columns)
df_party_series = df_party_series.rename(columns={"variable":"partido", "value":"votos"})
#df_party_series = df_party_series[df_party_series["votos"] > 0]
top_parties = list(df_party_series.groupby("partido")
                     .sum("votos").sort_values("votos", ascending=False)[:10]
                     .reset_index()["partido"].unique())
df_party_series = df_party_series[df_party_series["partido"].isin(top_parties)]
df_party_series = df_party_series.sort_values(["year", "month", "partido"], ascending=True)
#df_party_series["elecciones"] = df["year"].astype(str).str.zfill(2) + df["month"].astype(str).str.zfill(2)

In [ ]:
max_y = int(df_party_series["votos"].max())
fig = px.scatter(df_party_series, x="year", y="votos", animation_frame="year",animation_group="partido",
                 color="partido", range_x=[1976,2020], range_y=[0,max_y], color_discrete_map=color_map,
                 title= f"Elecciones Generales a lo largo del tiempo en {nombre_municipio}")
fig.show()

In [ ]:
fig = px.line(df_party_series, x="year", y="votos", color="partido", line_group="partido", hover_name="partido",
        line_shape="linear", render_mode="svg", color_discrete_map=color_map,range_x=[1976,2020],
        title= f"Elecciones Generales a lo largo del tiempo en {nombre_municipio}")
fig.show()